# 🎙️ SLAM-ASR: Speech Recognition with Speech LLM (LibriSpeech Evaluation)

This notebook demonstrates how to run **SLAM-ASR** (from the paper *[SLAM-LLM: Speech-Language Critical Evaluation](https://arxiv.org/abs/2402.08846)*) to perform Speech-to-Text (ASR) on the LibriSpeech dataset and reproduce the paper's claimed performance metrics.

## 📊 Paper Performance Benchmarks on LibriSpeech
| Speech Encoder | Projector | LLM | test-clean WER (%) | test-other WER (%) |
|---|---|---|---|---|
| **WavLM-large** | Linear (~18.88M) | Vicuna-7B-v1.5 | **2.28%** | 4.78% |
| **HuBERT-xtralarge** | Linear (~21.50M) | Vicuna-7B-v1.5 | **1.84%** | 3.39% |

---
### 🚀 Steps Covered in this Notebook:
1. **Environment Setup & Dependency Installation**
2. **Download Pre-trained Projector Weights & Speech Encoders**
3. **Prepare LibriSpeech Test Dataset & Manifest (`.jsonl`)**
4. **Run Batch Inference / Decoding using SLAM-LLM**
5. **Evaluate Word Error Rate (WER)**


## Step 1: GPU Check & Environment Setup

In [ ]:
# Verify GPU access (T4 / L4 / A100)
!nvidia-smi

In [ ]:
# Setup environment if running directly in Colab
import os
import sys

if not os.path.exists('src'):
    !git clone https://github.com/X-LANCE/SLAM-LLM.git slam_repo
    %cd slam_repo

# Ensure current path and src are in Python path and install package
!pip install -q -e .
!pip install -q torch torchaudio transformers>=4.31.0 accelerate bitsandbytes peft hydra-core>=1.3.2 omegaconf openai-whisper soundfile librosa jiwer gdown scipy sentencepiece optimum

## Step 2: Download Model Weights & Projector Checkpoints

We will download:
1. **WavLM-Large encoder weights** (`WavLM-Large.pt`)
2. **Trained Linear Projector weights** (~18.88M params)
3. **Vicuna-7B-v1.5 LLM** (loaded directly via HuggingFace `lmsys/vicuna-7b-v1.5`)

In [ ]:
import os
import gdown

os.makedirs('checkpoints', exist_ok=True)

# 1. Download WavLM-Large encoder checkpoint
wavlm_ckpt_path = 'checkpoints/WavLM-Large.pt'
if not os.path.exists(wavlm_ckpt_path):
    print('Downloading WavLM-Large checkpoint...')
    !wget -q -O {wavlm_ckpt_path} https://github.com/microsoft/unilm/raw/master/wavlm/WavLM-Large.pt || \
    python3 -c "import gdown; gdown.download('https://drive.google.com/uc?id=12-cB34qCTvByWT-QtOcZaqwwO21FLSqU', '{wavlm_ckpt_path}', quiet=False)"

# 2. Download Pre-trained Linear Projector checkpoint (~18.88M params)
projector_ckpt_path = 'checkpoints/wavlm_linear_projector.pt'
if not os.path.exists(projector_ckpt_path):
    print('Downloading Linear Projector checkpoint...')
    gdown.download('https://drive.google.com/uc?id=1cLNuMR05oXxKj8M_Z3yAZ5JHJ06ybIHp', projector_ckpt_path, quiet=False)

print('Checkpoints ready:')
!ls -lh checkpoints/

## Step 3: Prepare LibriSpeech Test Dataset & Manifest

We will download LibriSpeech `test-clean.tar.gz` and generate the JSONL data manifest in the format expected by SLAM-LLM:
```json
{"key": "1001-134707-0000_ASR", "source": "/path/to/audio.flac", "target": "TRANSCRIPTION HERE"}
```

In [ ]:
import os
import json
import tarfile
import urllib.request
from tqdm import tqdm

data_dir = 'data'
os.makedirs(data_dir, exist_ok=True)
tar_path = os.path.join(data_dir, 'test-clean.tar.gz')
extracted_dir = os.path.join(data_dir, 'LibriSpeech')

if not os.path.exists(extracted_dir):
    print('Downloading LibriSpeech test-clean (346 MB)...')
    url = 'https://www.openslr.org/resources/12/test-clean.tar.gz'
    urllib.request.urlretrieve(url, tar_path)
    print('Extracting dataset...')
    with tarfile.open(tar_path, 'r:gz') as tar:
        tar.extractall(path=data_dir)

# Create JSONL manifest
jsonl_path = os.path.join(data_dir, 'librispeech_test_clean.jsonl')
test_clean_root = os.path.join(extracted_dir, 'test-clean')

entries = []
for root, dirs, files in os.walk(test_clean_root):
    for file in files:
        if file.endswith('.trans.txt'):
            trans_file = os.path.join(root, file)
            with open(trans_file, 'r', encoding='utf-8') as tf:
                for line in tf:
                    parts = line.strip().split(' ', 1)
                    if len(parts) == 2:
                        utt_id, text = parts
                        audio_file = os.path.join(root, f"{utt_id}.flac")
                        if os.path.exists(audio_file):
                            entries.append({
                                "key": f"{utt_id}_ASR",
                                "source": os.path.abspath(audio_file),
                                "target": text
                            })

with open(jsonl_path, 'w', encoding='utf-8') as f:
    for entry in entries:
        f.write(json.dumps(entry) + '\n')

print(f'Generated manifest {jsonl_path} with {len(entries)} audio utterances.')

## Step 4: Run Batch Decoding / Inference

We run `inference_asr_batch.py` to evaluate SLAM-ASR (WavLM-Large + Vicuna-7B-v1.5) on LibriSpeech `test-clean`.

> **Note on GPU VRAM**: Vicuna-7B is loaded in half precision / 8-bit to fit smoothly inside Google Colab standard GPUs (T4 15GB VRAM).

In [ ]:
import sys
import torch

output_dir = 'output/decode_results'
os.makedirs(output_dir, exist_ok=True)
decode_log_prefix = os.path.join(output_dir, 'decode_test_clean_beam4')

# Run inference with PYTHONPATH set to include src/ and current dir
!PYTHONPATH=.:src:examples/asr_librispeech python3 examples/asr_librispeech/inference_asr_batch.py \
    --config-path "conf" \
    --config-name "prompt.yaml" \
    ++model_config.llm_name="vicuna-7b-v1.5" \
    ++model_config.llm_path="lmsys/vicuna-7b-v1.5" \
    ++model_config.llm_dim=4096 \
    ++model_config.encoder_name=wavlm \
    ++model_config.normalize=true \
    ++dataset_config.normalize=true \
    ++model_config.encoder_projector_ds_rate=5 \
    ++model_config.encoder_path="checkpoints/WavLM-Large.pt" \
    ++model_config.encoder_dim=1024 \
    ++model_config.encoder_projector=linear \
    ++dataset_config.dataset=speech_dataset \
    ++dataset_config.val_data_path="data/librispeech_test_clean.jsonl" \
    ++dataset_config.input_type=raw \
    ++dataset_config.inference_mode=true \
    ++train_config.model_name=asr \
    ++train_config.freeze_encoder=true \
    ++train_config.freeze_llm=true \
    ++train_config.batching_strategy=custom \
    ++train_config.val_batch_size=1 \
    ++train_config.output_dir={output_dir} \
    ++decode_log={decode_log_prefix} \
    ++ckpt_path="checkpoints/wavlm_linear_projector.pt"

## Step 5: Evaluate Word Error Rate (WER)

We calculate the Word Error Rate (WER) comparing model predictions (`decode_test_clean_beam4_pred`) with ground truth reference transcriptions (`decode_test_clean_beam4_gt`).

In [ ]:
import sys
import os
sys.path.insert(0, '.')
sys.path.insert(0, 'src')
sys.path.insert(0, 'examples/asr_librispeech')

try:
    from examples.asr_librispeech.eval_wer import calculate_wer
except ModuleNotFoundError:
    from eval_wer import calculate_wer

pred_file = f"{decode_log_prefix}_pred"
gt_file = f"{decode_log_prefix}_gt"
detail_file = f"{decode_log_prefix}_wer_detail.txt"

print("=== SLAM-ASR Evaluation Results ===")
calculate_wer(pred_file, gt_file, detail_file)


## 📌 Sample Predictions vs Ground Truth

Let me show some generated transcriptions compared with original ground truth sentences:

In [ ]:
with open(f"{decode_log_prefix}_pred") as f_pred, open(f"{decode_log_prefix}_gt") as f_gt:
    preds = [line.strip().split('\t', 1) for line in f_pred]
    gts = dict([line.strip().split('\t', 1) for line in f_gt])

print("Sample Transcriptions:")
print("=" * 80)
for key, pred_text in preds[:5]:
    gt_text = gts.get(key, 'N/A')
    print(f"[UTT]: {key}")
    print(f"  Ground Truth : {gt_text}")
    print(f"  Prediction   : {pred_text}")
    print("-" * 80)